# **64D, 128D & 256D Comparisons**


In [4]:
# =============================================================================
# URDU NEWS EMBEDDINGS WITH MULTI-DIMENSIONAL PCA COMPARISON
# Creates ChromaDB collections: Full (768D), PCA-64D, PCA-128D, PCA-256D
# Compares top 15 results and query latency across all dimensions
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import time
from sklearn.decomposition import PCA
import pickle
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import warnings
warnings.filterwarnings('ignore')

class MultiDimensionalUrduNewsEmbedder:
    """
    Generate embeddings for Urdu news with multiple PCA dimensions.
    Creates 4 collections: Full (768D), PCA-64D, PCA-128D, PCA-256D
    """

    def __init__(self, model_name: str = "urduhack/roberta-urdu-small",
                 base_path: str = "./chroma_db_collections",
                 pca_dimensions: list = [64, 128, 256]):
        """
        Initialize with multiple PCA dimensions.

        Args:
            model_name: HuggingFace model identifier
            base_path: Base path for all ChromaDB collections
            pca_dimensions: List of PCA dimensions to create
        """
        self.model_name = model_name
        self.base_path = Path(base_path)
        self.pca_dimensions = pca_dimensions
        self.pca_models = {}  # Dictionary to store PCA models for each dimension

        # Create base directory
        self.base_path.mkdir(exist_ok=True)

        # Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Initialize ChromaDB clients and collections
        self.clients = {}
        self.collections = {}

        # Full embeddings collection
        full_path = self.base_path / "chroma_db_full_768D"
        full_path.mkdir(exist_ok=True)
        print(f"Initializing Full Embeddings (768D) at: {full_path}")
        self.clients['full'] = chromadb.PersistentClient(path=str(full_path))
        self.collections['full'] = self.clients['full'].get_or_create_collection(
            name="urdu_news_full_768D",
            metadata={"hnsw:space": "cosine"}
        )

        # PCA collections
        for dim in pca_dimensions:
            pca_path = self.base_path / f"chroma_db_pca_{dim}D"
            pca_path.mkdir(exist_ok=True)
            print(f"Initializing PCA-{dim}D at: {pca_path}")
            self.clients[f'pca_{dim}'] = chromadb.PersistentClient(path=str(pca_path))
            self.collections[f'pca_{dim}'] = self.clients[f'pca_{dim}'].get_or_create_collection(
                name=f"urdu_news_pca_{dim}D",
                metadata={"hnsw:space": "cosine"}
            )

    def mean_pooling(self, model_output, attention_mask):
        """Apply mean pooling to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def generate_embedding_for_text(self, text: str, max_length: int = 512,
                                    chunk_overlap: int = 50) -> np.ndarray:
        """Generate embedding for text using mean pooling with chunking for long texts."""
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                text, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            return embeddings.cpu().detach().numpy()[0]

        # For long texts: chunk and average
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        return np.mean(chunk_embeddings, axis=0)

    def fit_pca_models(self, embeddings_array: np.ndarray) -> None:
        """Fit PCA models for all dimensions."""
        print(f"\n{'='*70}")
        print(f"FITTING PCA MODELS FOR DIMENSIONS: {self.pca_dimensions}")
        print(f"{'='*70}")
        print(f"Input shape: {embeddings_array.shape}")

        for dim in self.pca_dimensions:
            print(f"\nFitting PCA-{dim}D...")
            pca = PCA(n_components=dim, random_state=42)
            pca.fit(embeddings_array)
            self.pca_models[dim] = pca

            explained_var = np.sum(pca.explained_variance_ratio_) * 100
            print(f"  ✓ Explained variance: {explained_var:.2f}%")

            # Save PCA model
            pca_path = self.base_path / f"chroma_db_pca_{dim}D" / "pca_model.pkl"
            with open(pca_path, 'wb') as f:
                pickle.dump(pca, f)
            print(f"  ✓ Model saved to: {pca_path}")

    def apply_pca(self, embeddings_array: np.ndarray, dimension: int) -> np.ndarray:
        """Apply PCA transformation for specific dimension."""
        if dimension not in self.pca_models:
            raise ValueError(f"PCA model for {dimension}D not fitted.")
        return self.pca_models[dimension].transform(embeddings_array)

    def generate_embeddings_for_dataset(self, df: pd.DataFrame,
                                       content_column: str = "content",
                                       headline_column: str = "Headline",
                                       category_column: str = "Category") -> None:
        """Generate and store embeddings in all collections."""
        print(f"\n{'='*70}")
        print(f"GENERATING EMBEDDINGS FOR {len(df)} ARTICLES")
        print(f"{'='*70}")

        total_articles = len(df)
        start_time = time.time()

        ids = []
        embeddings_full = []
        metadatas = []
        documents = []

        # Step 1: Generate full embeddings
        print("\nSTEP 1: Generating full embeddings (768D)...")
        print("="*70)

        for idx, row in df.iterrows():
            if (idx + 1) % 500 == 0:
                elapsed = time.time() - start_time
                print(f"Processed {idx + 1}/{total_articles} articles ({elapsed:.2f}s)")

            content_text = str(row[content_column])
            if len(content_text.strip()) == 0:
                continue

            try:
                embedding = self.generate_embedding_for_text(content_text)

                ids.append(f"article_{idx}")
                embeddings_full.append(embedding)
                documents.append(content_text[:500])

                metadatas.append({
                    "article_index": idx,
                    "headline": str(row.get(headline_column, "Unknown")),
                    "category": str(row.get(category_column, "Unknown")),
                    "content_length": len(content_text)
                })
            except Exception as e:
                print(f"Error processing article {idx}: {str(e)}")
                continue

        embeddings_full_array = np.array(embeddings_full)

        # Step 2: Fit PCA models
        self.fit_pca_models(embeddings_full_array)

        # Step 3: Store full embeddings
        print(f"\n{'='*70}")
        print("STEP 3: Storing full embeddings (768D)...")
        print("="*70)
        self._store_embeddings('full', ids, embeddings_full_array, documents, metadatas, 768)

        # Step 4: Store PCA embeddings for all dimensions
        for dim in self.pca_dimensions:
            print(f"\n{'='*70}")
            print(f"STEP 4.{self.pca_dimensions.index(dim)+1}: Applying and storing PCA-{dim}D...")
            print("="*70)

            embeddings_pca = self.apply_pca(embeddings_full_array, dim)
            self._store_embeddings(f'pca_{dim}', ids, embeddings_pca, documents, metadatas, dim)

        total_time = time.time() - start_time
        print(f"\n{'='*70}")
        print("✓ ALL EMBEDDINGS GENERATED AND STORED!")
        print(f"{'='*70}")
        print(f"Total embeddings: {len(ids)}")
        print(f"Collections created:")
        print(f"  - Full (768D): {len(ids)} embeddings")
        for dim in self.pca_dimensions:
            print(f"  - PCA-{dim}D: {len(ids)} embeddings")
        print(f"Total time: {total_time:.2f}s ({total_time/60:.2f} min)")

    def _store_embeddings(self, collection_key: str, ids: list, embeddings: np.ndarray,
                         documents: list, metadatas: list, dimension: int) -> None:
        """Helper to store embeddings in batches."""
        batch_size = 5000
        collection = self.collections[collection_key]

        for batch_idx in range(0, len(ids), batch_size):
            batch_end = min(batch_idx + batch_size, len(ids))
            print(f"  Storing items {batch_idx} to {batch_end}...")

            batch_metadatas = [
                {**meta, "embedding_type": collection_key, "dimensions": dimension}
                for meta in metadatas[batch_idx:batch_end]
            ]

            collection.add(
                ids=ids[batch_idx:batch_end],
                embeddings=[emb.tolist() for emb in embeddings[batch_idx:batch_end]],
                documents=documents[batch_idx:batch_end],
                metadatas=batch_metadatas
            )
        print(f"  ✓ Stored successfully!")

    def search_with_latency(self, query_text: str, collection_key: str,
                           n_results: int = 15) -> tuple:
        """Search and measure query latency."""
        # Generate query embedding
        query_embedding = self.generate_embedding_for_text(query_text)

        # Apply PCA if needed
        if collection_key.startswith('pca_'):
            dim = int(collection_key.split('_')[1])
            if dim not in self.pca_models:
                # Load PCA model
                pca_path = self.base_path / f"chroma_db_pca_{dim}D" / "pca_model.pkl"
                with open(pca_path, 'rb') as f:
                    self.pca_models[dim] = pickle.load(f)
            query_embedding = self.pca_models[dim].transform(query_embedding.reshape(1, -1))[0]

        # Measure query latency
        start_time = time.time()
        results = self.collections[collection_key].query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results
        )
        latency = (time.time() - start_time) * 1000  # Convert to milliseconds

        return results, latency

    def compare_results_comprehensive(self, queries: list, output_dir: str = "./comparison_results"):
        """Comprehensive comparison of all embeddings with top 15 results."""
        output_path = Path(output_dir)
        output_path.mkdir(exist_ok=True)

        print(f"\n{'='*70}")
        print("COMPREHENSIVE COMPARISON: TOP 15 RESULTS ACROSS ALL DIMENSIONS")
        print(f"{'='*70}")

        all_comparisons = []

        for query_idx, query in enumerate(queries, 1):
            print(f"\n{'='*70}")
            print(f"QUERY {query_idx}: {query[:80]}...")
            print(f"{'='*70}")

            # Search in all collections
            results_full, latency_full = self.search_with_latency(query, 'full', n_results=15)
            ids_full = set(results_full['ids'][0])

            print(f"\n✓ Full (768D) - Latency: {latency_full:.2f}ms")
            print(f"  Top 5 results:")
            for i in range(min(5, len(results_full['ids'][0]))):
                headline = results_full['metadatas'][0][i].get('headline', 'N/A')
                score = 1 - results_full['distances'][0][i]
                print(f"    {i+1}. [{score:.4f}] {headline[:60]}...")

            comparison_data = {
                'query': query,
                'query_idx': query_idx,
                'full_latency': latency_full,
                'full_ids': ids_full,
                'pca_results': {}
            }

            # Compare with each PCA dimension
            for dim in self.pca_dimensions:
                results_pca, latency_pca = self.search_with_latency(
                    query, f'pca_{dim}', n_results=15
                )
                ids_pca = set(results_pca['ids'][0])

                overlap = len(ids_full.intersection(ids_pca))
                overlap_pct = (overlap / 15) * 100

                print(f"\n✓ PCA-{dim}D - Latency: {latency_pca:.2f}ms")
                print(f"  Overlap with Full: {overlap}/15 ({overlap_pct:.1f}%)")
                print(f"  Top 5 results:")
                for i in range(min(5, len(results_pca['ids'][0]))):
                    headline = results_pca['metadatas'][0][i].get('headline', 'N/A')
                    score = 1 - results_pca['distances'][0][i]
                    print(f"    {i+1}. [{score:.4f}] {headline[:60]}...")

                comparison_data['pca_results'][dim] = {
                    'latency': latency_pca,
                    'ids': ids_pca,
                    'overlap': overlap,
                    'overlap_pct': overlap_pct
                }

            all_comparisons.append(comparison_data)

            # Create comparison pie chart for this query
            self._create_comparison_pie_chart(comparison_data, output_path)

        # Create summary visualizations
        self._create_summary_visualizations(all_comparisons, output_path)

        return all_comparisons

    def _create_comparison_pie_chart(self, comparison_data: dict, output_path: Path):
        """Create overlapping pie charts for one query."""
        query_idx = comparison_data['query_idx']

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"Query {query_idx}: Overlap with Full (768D) - Top 15 Results",
                     fontsize=14, fontweight='bold')

        for idx, dim in enumerate(self.pca_dimensions):
            ax = axes[idx]
            pca_data = comparison_data['pca_results'][dim]

            overlap = pca_data['overlap']
            non_overlap = 15 - overlap

            # Create pie chart
            sizes = [overlap, non_overlap]
            colors = ['#2ecc71', '#e74c3c']
            labels = [f'Overlap: {overlap}/15\n({pca_data["overlap_pct"]:.1f}%)',
                     f'Different: {non_overlap}/15']
            explode = (0.05, 0)

            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
                  startangle=90, explode=explode, textprops={'fontsize': 10})
            ax.set_title(f'Full vs PCA-{dim}D\nLatency: {pca_data["latency"]:.2f}ms',
                        fontsize=12, fontweight='bold')

        plt.tight_layout()
        plt.savefig(output_path / f'query_{query_idx}_comparison.png', dpi=300, bbox_inches='tight')
        plt.close()

        print(f"  ✓ Saved: query_{query_idx}_comparison.png")

    def _create_summary_visualizations(self, all_comparisons: list, output_path: Path):
        """Create summary visualizations across all queries."""
        print(f"\n{'='*70}")
        print("CREATING SUMMARY VISUALIZATIONS")
        print(f"{'='*70}")

        # Calculate averages
        avg_latencies = {'full': np.mean([c['full_latency'] for c in all_comparisons])}
        avg_overlaps = {}

        for dim in self.pca_dimensions:
            latencies = [c['pca_results'][dim]['latency'] for c in all_comparisons]
            overlaps = [c['pca_results'][dim]['overlap_pct'] for c in all_comparisons]
            avg_latencies[f'pca_{dim}'] = np.mean(latencies)
            avg_overlaps[dim] = np.mean(overlaps)

        # Create summary figure
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Plot 1: Average Query Latency
        ax1 = axes[0]
        dims = ['Full\n(768D)'] + [f'PCA-{d}D' for d in self.pca_dimensions]
        latencies = [avg_latencies['full']] + [avg_latencies[f'pca_{d}'] for d in self.pca_dimensions]
        colors = ['#3498db', '#e67e22', '#9b59b6', '#1abc9c']

        bars = ax1.bar(dims, latencies, color=colors, edgecolor='black', linewidth=1.5)
        ax1.set_ylabel('Average Latency (ms)', fontsize=12, fontweight='bold')
        ax1.set_title('Average Query Latency Comparison', fontsize=13, fontweight='bold')
        ax1.grid(axis='y', alpha=0.3, linestyle='--')

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}ms', ha='center', va='bottom', fontsize=10, fontweight='bold')

        # Plot 2: Average Overlap Percentage
        ax2 = axes[1]
        dims_pca = [f'PCA-{d}D' for d in self.pca_dimensions]
        overlaps = [avg_overlaps[d] for d in self.pca_dimensions]
        colors_pca = ['#e67e22', '#9b59b6', '#1abc9c']

        bars = ax2.bar(dims_pca, overlaps, color=colors_pca, edgecolor='black', linewidth=1.5)
        ax2.set_ylabel('Average Overlap (%)', fontsize=12, fontweight='bold')
        ax2.set_title('Average Overlap with Full Embeddings (Top 15)', fontsize=13, fontweight='bold')
        ax2.set_ylim([0, 100])
        ax2.grid(axis='y', alpha=0.3, linestyle='--')
        ax2.axhline(y=100, color='green', linestyle='--', linewidth=2, alpha=0.5, label='100% Overlap')
        ax2.legend()

        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

        plt.tight_layout()
        plt.savefig(output_path / 'summary_comparison.png', dpi=300, bbox_inches='tight')
        plt.close()

        print("✓ Saved: summary_comparison.png")

        # Print summary statistics
        print(f"\n{'='*70}")
        print("SUMMARY STATISTICS")
        print(f"{'='*70}")
        print(f"\nAverage Query Latencies:")
        print(f"  Full (768D): {avg_latencies['full']:.2f}ms")
        for dim in self.pca_dimensions:
            print(f"  PCA-{dim}D: {avg_latencies[f'pca_{dim}']:.2f}ms " +
                  f"({(avg_latencies[f'pca_{dim}']/avg_latencies['full']*100):.1f}% of Full)")

        print(f"\nAverage Overlap with Full (Top 15):")
        for dim in self.pca_dimensions:
            print(f"  PCA-{dim}D: {avg_overlaps[dim]:.1f}%")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*70)
    print("MULTI-DIMENSIONAL PCA URDU NEWS EMBEDDINGS")
    print("Dimensions: Full (768D), PCA-64D, PCA-128D, PCA-256D")
    print("="*70)

    # Load dataset with proper encoding handling
    print("\nLoading dataset...")
    try:
        # Try UTF-8 first
        df = pd.read_csv("final_cleaned_urdu_news.csv", encoding='utf-8')
        print("✓ Dataset loaded with UTF-8 encoding")
    except UnicodeDecodeError:
        try:
            # Try common encodings for Urdu text
            encodings_to_try = ['utf-8-sig', 'cp1256', 'iso-8859-1', 'latin1', 'windows-1256']
            for encoding in encodings_to_try:
                try:
                    df = pd.read_csv("final_cleaned_urdu_news.csv", encoding=encoding)
                    print(f"✓ Dataset loaded with {encoding} encoding")
                    break
                except UnicodeDecodeError:
                    continue
            else:
                # If all encodings fail, try with error handling
                df = pd.read_csv("final_cleaned_urdu_news.csv", encoding='utf-8', errors='ignore')
                print("✓ Dataset loaded with error handling (some characters may be lost)")
        except Exception as e:
            print(f"Error loading dataset: {e}")
            raise

    print(f"✓ Dataset loaded: {df.shape[0]} articles")
    print(f"  Categories: {df['Category'].unique().tolist()}")

    # Initialize embedder
    print(f"\n{'='*70}")
    print("INITIALIZING MULTI-DIMENSIONAL EMBEDDER")
    print(f"{'='*70}")

    embedder = MultiDimensionalUrduNewsEmbedder(
        model_name="urduhack/roberta-urdu-small",
        base_path="./chroma_db_collections",
        pca_dimensions=[64, 128, 256]
    )

    # Generate embeddings (comment out if already generated)
    embedder.generate_embeddings_for_dataset(
        df=df,
        content_column="content",
        headline_column="Headline",
        category_column="Category"
    )

    # Load test queries from file with proper encoding
    print(f"\n{'='*70}")
    print("LOADING TEST QUERIES")
    print(f"{'='*70}")

    try:
        with open('test_queries.txt', 'r', encoding='utf-8') as file:
            content = file.read()

        # Extract the test_queries list
        test_queries = eval(content.split('test_queries = ')[1])
        print(f"✓ Loaded {len(test_queries)} test queries")

        # Truncate queries to 150 characters
        # --- TRUNCATION POINT: Change the number 150 below to modify character limit ---
        test_queries = [query[:150] for query in test_queries]
        # --- END TRUNCATION POINT ---
        print(f"✓ Truncated all queries to 150 characters")

    except Exception as e:
        print(f"Error loading test queries: {e}")
        print("Using default test queries instead")
        test_queries = [
        "معیشت اور کاروبار کی خبریں",
        "کرکٹ کی تازہ ترین خبریں",
        "فلموں اور ڈرامے کی خبریں",
        "پاکستان کا انتخابی نظام",
        "پاکستانی شوبز انڈسٹری",
        "صحت اور تندرستی کے حوالے سے مفید معلومات",
        "پاکستان میں تعلیمی نظام اور جدید تربیت",
        "ٹیکنالوجی",
        "کاروبار اور معاشی ترقی کی خبریں",
        "پاکستانی سیاست اور حکومتی پالیسیاں",
        "کھیلوں اور تفریحی پروگراموں کی خبریں",
        "مذہبی تعلیمات اور روحانی معلومات",
        "پاکستان کے خوبصورت سیاحتی مقامات"
     ]

    # Run comprehensive comparison
    print(f"\n{'='*70}")
    print("STARTING COMPREHENSIVE COMPARISON")
    print(f"{'='*70}")

    comparisons = embedder.compare_results_comprehensive(
        queries=test_queries,
        output_dir="./comparison_results"
    )

    print(f"\n{'='*70}")
    print("✓ ALL COMPARISONS COMPLETED!")
    print(f"{'='*70}")
    print(f"Results saved in: ./comparison_results/")
    print(f"  - Individual query comparisons: query_1_comparison.png, etc.")
    print(f"  - Summary statistics: summary_comparison.png")

MULTI-DIMENSIONAL PCA URDU NEWS EMBEDDINGS
Dimensions: Full (768D), PCA-64D, PCA-128D, PCA-256D
MODIFIED: Stores content in metadata

Loading dataset...
✓ Dataset loaded with UTF-8 encoding
✓ Dataset loaded: 111853 articles
  Categories: ['Business & Economics', 'Entertainment', 'Science & Technology', 'Sports', nan]

INITIALIZING MULTI-DIMENSIONAL EMBEDDER
Using device: cuda
Loading model: urduhack/roberta-urdu-small
Initializing Full Embeddings (768D) at: chroma_db_collections/chroma_db_full_768D
Initializing PCA-64D at: chroma_db_collections/chroma_db_pca_64D
Initializing PCA-128D at: chroma_db_collections/chroma_db_pca_128D
Initializing PCA-256D at: chroma_db_collections/chroma_db_pca_256D

GENERATING EMBEDDINGS FOR 111853 ARTICLES

STEP 1: Generating full embeddings (768D)...
Error processing article 0: name 'Headline' is not defined
Error processing article 1: name 'Headline' is not defined
Error processing article 2: name 'Headline' is not defined
Error processing article 3: nam

KeyboardInterrupt: 

# **Recommender Code PCA Grok**

In [3]:
# =============================================================================
# URDU NEWS RECOMMENDATION SYSTEM WITH 128D PCA EMBEDDINGS
# Retrieves content directly from ChromaDB metadata
# Displays headline + article content (minimum 250 characters)
# OUTPUTS RESULTS TO WORD DOCUMENT AND CONSOLE
# =============================================================================

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import chromadb
from pathlib import Path
import pickle
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


class UrduNewsRecommender128D:
    """
    Recommendation system using 128D PCA-reduced embeddings.
    Retrieves article content directly from ChromaDB metadata.
    Displays headline and article text (minimum 250 characters).
    """

    def __init__(self,
                 model_name: str = "urduhack/roberta-urdu-small",
                 chroma_db_path: str = "chroma_db_collections/chroma_db_pca_128D",
                 collection_name: str = "urdu_news_pca_128D",
                 output_doc_path: str = "Urdu_News_128D_Recommendations.docx"):
        """
        Initialize recommender with 128D PCA embeddings.

        Args:
            model_name: HuggingFace model identifier
            chroma_db_path: Path to ChromaDB with 128D PCA embeddings
            collection_name: Name of the collection
            output_doc_path: Path for the output Word document
        """
        self.model_name = model_name
        self.chroma_db_path = Path(chroma_db_path)
        self.pca_model = None
        self.output_doc_path = output_doc_path

        # Initialize Word document
        self.doc = Document()
        self.setup_document()

        # Check if GPU is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.add_heading(f"Device Information: {self.device}", level=2)

        # Load model and tokenizer
        print(f"Loading model: {model_name}")
        self.add_paragraph(f"Loading model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()

        # Load PCA model (128D)
        pca_model_path = self.chroma_db_path / "pca_model.pkl"
        if pca_model_path.exists():
            print(f"Loading PCA model from: {pca_model_path}")
            self.add_paragraph(f"Loading PCA model from: {pca_model_path}")
            with open(pca_model_path, 'rb') as f:
                self.pca_model = pickle.load(f)
            print(f"✓ PCA model loaded (128 dimensions)")
            self.add_paragraph(f"✓ PCA model loaded (128 dimensions)")
        else:
            raise FileNotFoundError(f"PCA model not found at {pca_model_path}")

        # Connect to ChromaDB
        print(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.add_paragraph(f"Connecting to ChromaDB at: {self.chroma_db_path}")
        self.client = chromadb.PersistentClient(path=str(self.chroma_db_path))

        try:
            self.collection = self.client.get_collection(name=collection_name)
            print(f"✓ Connected to collection: {collection_name}")
            print(f"✓ Total articles in database: {self.collection.count()}\n")
            self.add_paragraph(f"✓ Connected to collection: {collection_name}")
            self.add_paragraph(f"✓ Total articles in database: {self.collection.count()}")
        except Exception as e:
            print(f"Error: Could not find collection '{collection_name}'")
            raise e

    def setup_document(self):
        """Setup the Word document with title and basic formatting."""
        # Add title
        title = self.doc.add_heading('Urdu News Recommendation System\n128D PCA Embeddings - Content from Metadata', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add timestamp
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        time_para = self.doc.add_paragraph(f"Report Generated on: {current_time}")
        time_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Add empty line
        self.doc.add_paragraph()

    def add_heading(self, text, level=1):
        """Add heading to document."""
        heading = self.doc.add_heading(text, level)
        return heading

    def add_paragraph(self, text='', style=None, bold=False, color=None):
        """Add paragraph to document with optional formatting."""
        para = self.doc.add_paragraph()
        run = para.add_run(text)
        
        if bold:
            run.bold = True
        if color:
            run.font.color.rgb = color
            
        if style:
            para.style = style
        return para

    def save_document(self):
        """Save the Word document."""
        self.doc.save(self.output_doc_path)
        print(f"\n✓ Word document saved to: {self.output_doc_path}")

    def mean_pooling(self, model_output, attention_mask):
        """Apply MEAN POOLING to get sentence embeddings."""
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

    def generate_query_embedding(self, query_text: str, max_length: int = 512,
                                 chunk_overlap: int = 50) -> np.ndarray:
        """
        Generate 128D PCA embedding for query text using MEAN POOLING.
        """
        tokens = self.tokenizer.encode(query_text, add_special_tokens=True)

        # Process short queries
        if len(tokens) <= max_length:
            encoded_input = self.tokenizer(
                query_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embedding = embeddings.cpu().detach().numpy()[0]

            # Apply PCA to reduce to 128D
            if self.pca_model is not None:
                embedding = self.pca_model.transform(embedding.reshape(1, -1))[0]

            return embedding

        # For long queries: use chunking
        chunk_size = max_length - 2
        stride = chunk_size - chunk_overlap
        chunk_embeddings = []

        for i in range(0, len(tokens), stride):
            chunk_tokens = tokens[i:i + chunk_size]
            if len(chunk_tokens) < 50:
                break

            chunk_text = self.tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            encoded_input = self.tokenizer(
                chunk_text,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            )
            encoded_input = {k: v.to(self.device) for k, v in encoded_input.items()}

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            chunk_embedding = self.mean_pooling(model_output, encoded_input['attention_mask'])
            chunk_embeddings.append(chunk_embedding.cpu().detach().numpy()[0])

        final_embedding = np.mean(chunk_embeddings, axis=0)

        # Apply PCA to reduce to 128D
        if self.pca_model is not None:
            final_embedding = self.pca_model.transform(final_embedding.reshape(1, -1))[0]

        return final_embedding

    def get_recommendations(self, query: str, n_results: int = 10,
                          filter_category: str = None) -> dict:
        """
        Get recommendations using 128D PCA embeddings.
        """
        print(f"\n{'='*80}")
        print(f"GENERATING RECOMMENDATIONS (128D PCA)")
        print(f"{'='*80}")
        print(f"Query: {query[:100]}...")
        print(f"Number of results: {n_results}")
        if filter_category:
            print(f"Category filter: {filter_category}")
        print(f"{'='*80}\n")

        # Build where clause for category filtering
        where_clause = None
        if filter_category:
            where_clause = {"category": filter_category}

        # Generate 128D PCA embedding
        print("→ Generating query embedding (128D PCA with Mean Pooling)...")
        query_embedding = self.generate_query_embedding(query)
        print(f"  ✓ Embedding generated: shape {query_embedding.shape}")

        # Search in ChromaDB
        print("→ Searching in database...")
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=n_results,
            where=where_clause
        )
        print(f"  ✓ Found {len(results['ids'][0])} recommendations\n")

        return results

    def get_article_content(self, metadata: dict, min_chars: int = 250) -> str:
        """
        Extract article content from metadata (minimum 250 characters).

        Args:
            metadata: Article metadata dictionary
            min_chars: Minimum characters to display (default 250)

        Returns:
            Article content (at least min_chars if available)
        """
        # Try to get content from metadata
        content = metadata.get('content', '')
        
        # If content is empty, try alternative fields
        if not content or content == 'N/A':
            description = metadata.get('description', '')
            text = metadata.get('text', '')
            
            # Combine available text fields
            content = f"{description}\n\n{text}".strip()
        
        # If still no content
        if not content or len(content.strip()) == 0:
            return "محتویٰ دستیاب نہیں (Content not available)"
        
        # Return at least min_chars characters (or full content if shorter)
        if len(content) < min_chars:
            return content  # Return what we have if less than min_chars
        
        return content[:min_chars]  # Return exactly min_chars characters

    def display_recommendations(self, results: dict, query: str, query_number: int = 1):
        """
        Display detailed recommendations with headline and article content from metadata.
        Outputs to both console and Word document.
        """
        if not results['ids'] or len(results['ids'][0]) == 0:
            print("❌ No recommendations found!")
            self.add_paragraph("❌ No recommendations found!")
            return

        # Extract data
        ids = results['ids'][0]
        distances = results['distances'][0]
        metadatas = results['metadatas'][0]

        # Convert distances to similarity scores
        similarities = [1 - d for d in distances]

        # Add query heading to document
        self.doc.add_page_break()
        self.add_heading(f"Query {query_number}: Recommendations", level=1)
        
        # Add query text
        query_para = self.doc.add_paragraph()
        query_para.add_run("Query: ").bold = True
        query_para.add_run(query)
        self.add_paragraph(f"Top {len(ids)} recommendations\n")

        # Console output
        print(f"\n{'='*80}")
        print(f"QUERY {query_number} RECOMMENDATIONS")
        print(f"{'='*80}")
        print(f"Query: {query}\n")
        print(f"{'='*80}\n")

        # Display each recommendation
        for i, (article_id, similarity, metadata) in enumerate(zip(ids, similarities, metadatas), 1):
            # Get article details from metadata
            headline = metadata.get('headline', 'عنوان دستیاب نہیں')
            category = metadata.get('category', 'N/A')
            article_idx = metadata.get('article_index', 'Unknown')
            
            # Get content (minimum 250 characters)
            content = self.get_article_content(metadata, min_chars=250)
            content_length = len(content)

            # Console output
            print(f"{'─'*80}")
            print(f"RECOMMENDATION #{i}")
            print(f"{'─'*80}")
            print(f"Article ID: {article_id}")
            print(f"Article Index: {article_idx}")
            print(f"Similarity Score: {similarity:.4f}")
            print(f"Category: {category}")
            print(f"\nHeadline:")
            print(f"{headline}")
            print(f"\nArticle Content ({content_length} characters):")
            print(f"{content}")
            print(f"{'─'*80}\n")

            # Word document output
            self.add_heading(f"Recommendation #{i}", level=2)
            
            # Add details in a formatted way
            details_para = self.doc.add_paragraph()
            details_para.add_run(f"Article ID: ").bold = True
            details_para.add_run(f"{article_id}\n")
            details_para.add_run(f"Article Index: ").bold = True
            details_para.add_run(f"{article_idx}\n")
            details_para.add_run(f"Similarity Score: ").bold = True
            details_para.add_run(f"{similarity:.4f}\n")
            details_para.add_run(f"Category: ").bold = True
            details_para.add_run(f"{category}\n\n")
            
            # Add headline
            headline_para = self.doc.add_paragraph()
            headline_para.add_run("Headline:\n").bold = True
            headline_para.add_run(headline)
            headline_para.add_run("\n\n")
            
            # Add article content
            content_para = self.doc.add_paragraph()
            content_para.add_run(f"Article Content ({content_length} characters):\n").bold = True
            content_para.add_run(content)
            
            self.add_paragraph()  # Empty line between recommendations

    def add_statistics_to_doc(self):
        """Add system statistics to the Word document."""
        self.add_heading("System Statistics", level=1)
        
        explained_var = np.sum(self.pca_model.explained_variance_ratio_) * 100 if self.pca_model else 0
        
        stats_data = [
            ["Model", self.model_name],
            ["Device", str(self.device)],
            ["Pooling Method", "MEAN POOLING"],
            ["Embedding Dimension", "128 (PCA Reduced)"],
            ["PCA Explained Variance", f"{explained_var:.2f}%"],
            ["Total Articles", self.collection.count()],
            ["Collection Name", self.collection.name],
            ["Content Source", "ChromaDB Metadata"]
        ]
        
        # Create table in document
        table = self.doc.add_table(rows=1, cols=2)
        table.style = 'Light Grid Accent 1'
        
        # Add headers
        header_cells = table.rows[0].cells
        header_cells[0].text = "Parameter"
        header_cells[1].text = "Value"
        
        # Add data rows
        for param, value in stats_data:
            row_cells = table.add_row().cells
            row_cells[0].text = str(param)
            row_cells[1].text = str(value)
        
        self.add_paragraph()

        # Console output
        print("\n" + "="*80)
        print("SYSTEM STATISTICS")
        print("="*80)
        for param, value in stats_data:
            print(f"{param}: {value}")
        print("="*80 + "\n")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("="*80)
    print("URDU NEWS RECOMMENDATION SYSTEM")
    print("128D PCA Embeddings with Mean Pooling")
    print("Content Retrieved from ChromaDB Metadata")
    print("="*80)

    # Initialize recommender
    print("\n" + "="*80)
    print("INITIALIZING RECOMMENDATION SYSTEM")
    print("="*80 + "\n")

    recommender = UrduNewsRecommender128D(
        model_name="urduhack/roberta-urdu-small",
        chroma_db_path="chroma_db_collections/chroma_db_pca_128D",
        collection_name="urdu_news_pca_128D",
        output_doc_path="Urdu_News_128D_Recommendations.docx"
    )

    # Add system statistics
    recommender.add_statistics_to_doc()

    # Define your queries here - paste them below
    test_queries = [
    "ریاضی اس سوال جواب دے پہلی نظر ریاضی سان سوال لگتا مگر اس انٹرنیٹ متعدد افراد ذہنوں پریشان کرکے رکھ ہםکیا اپنی ریاضی صلاحیت پورا بھروسا ہاں اس سوال جو",
    "پاکستان اسٹاک ایکسچینج ملا جلا رجحان ہنڈرڈ انڈیکس پوائنٹس کمی بند پاکستان اسٹاک ایکسچینج اج کاروبار اختتام ملا جلا رجحان دیکھا گیا پاکستان اسٹاک ایکسچ",
    "سام سنگ نئے فلیگ شپ فون تاریخ رونمائی سامنے گئی سام سنگ اپنے نئے فلیگ شپ فون گلیکسی نوٹ ئندہ ماہ متعارف کرانے باضابطہ اعلان کردیا ہے سام سنگ جانب اگست",
    "سلمان خان دوستوں دوستی لے ڈوبی ممبئی ویب ڈیسک بالی وڈ پنڈتوں کہنا بظاہر نظر پروڈکشن میدان اترنے سلمان خان پہلی بار جھٹکا لگنے اپنے دوستوں بچوں عطیہ شی",
    "ئی فون متعارف کرانے تاریخ سامنے گئی رواں سال شروع ایسی اطلاعات سامنے ئی تھیں ئی فون ایٹ اس بار معمول ستمبر سامنے نہیں سکے بلکہ اسے پیش کیا جائے گا تاہ",
    "پاکستان ویسٹ انڈیز قسمت بدلنے خواہاں برج ٹان پاکستان عالمی چیمپیئن ویسٹ انڈیز درمیان چار ٹی میچوں سیریز اج اغاز ہونے جا جہاں میزبان ٹیم متحدہ عرب امار",
    "ایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم کراچی ایشین اسنوکر چمپئن شپ پاکستانی کیوسٹ سفر ختم ہوگیا کوارٹر فائنل شکست بعد محمد سجاد ابو صائم ایونٹ ہ",
    "یو ایس اوپن ٹینس نوواک جوکووچ تیسرے رانڈ پہنچ گئے نیویارک اگست یو ایس اوپن ٹاپ سیڈ دفاعی چیمپئن نوواک جوکووچ باسانی تیسرے رانڈ پہنچ گئے اکہترویں فتح پ",
    "پی ٹی اے زونگ جی اشتہارات واپس لینے ہدایت پاکستان ٹیلی کمیونیکشن اتھارٹی پی ٹی اے چائنا موبائل پاکستان سی پی ایم یعنی زونگ جی ٹیکنالوجی اشتہارات تشہیر",
    "ناقدین نظر بہترین فلمیں ناقدین نظر بہترین فلمیں امریکن گریفیٹی سائیکو سم لائیک اٹ ہاٹ منتھس ویکس اینڈ ڈیز گون ود ونڈ مائی لیفٹ فٹ ہوپ ڈریمز پینز لیبیر",
    "پاکستانی اسکواڈ کرائسٹ چرچ کوئنز ٹان روانہ ہو لاہور دسمبر نیوزی لینڈ موجود قومی کرکٹ اسکواڈ مینیجڈ ئسولیشن چھوڑنے اجازت مل گئی قومی اسکواڈ شامل اکاون ",
    "سرچ ہونے ڈیوائسز پسند ہو نہ ہو مگر ایپل ئی فون سکس ایس گوگل مقبول ڈیوائس قرار دے ہے گوگل جانب ٹرینڈز ٹیکنالوجی فہرست ئی فون سکس ایس سرچ ہونے باعث سرفہ",
    "پاکستان اسٹاک ایکسچینج پوائنٹس اضافہ کراچی اپریل کاروباری ہفتے تیسرے روز پاکستان اسٹاک ایکسچینج مثبت رجحان انڈیکس پوائنٹس بہتری بدھ پاکستان اسٹاک ایکس",
    "پاکستان ایم بی رفتار انٹرنیٹ فراہمی ممکن ہوگئی ٹرانس ورلڈ کمپنی دعوی کیا پاکستان چالیس پچاس ایم بی انٹرنیٹ سروس فراہم جاسکے سی می وی فائیو اپنی بیس ہز",
    "حکومت تیل قیمتوں کمی فائدہ اٹھانے فیصلہ اسلام اباد مئی حکومت عالمی منڈی تیل قیمتوں کمی فائدہ اٹھانے فیصلہ کیا ملکی تاریخ پہلی بار دو سال تیل پیشگی خری",
    "راہول ڈریوڈ انٹرنیشنل کرکٹ ریٹائرمنٹ اعلان کردیا بنگلور بھارتی اسٹار بیٹسمین راہول ڈریوڈ انٹرنیشنل کرکٹ ریٹائرمنٹ کا اعلان کردیا بنگلور پریس کانفرنس د",
    "جنوبی کوریا پاکستان تانبا معدنی اشیاء درامد کرے اسلام اباد جنوری کوریا پاکستان خام مال درامد کرنے کیلئے متعلقہ حکام وفو دکی سطح باقاعدہ ملاقاتوں سلسلہ",
    "ایل پی جی قیمت جون روپے فی کلو کمی اعلان کراچی ایل پی جی عالمی قیمت ڈالر فی ٹن کمی بعد جون مقامی سطح ایل پی جی قیمت روپے فی کلو کمی ہو جائے ایل پی جی ",
    "فلائی ویٹ چیمپئن محمد وسیم اٹھویں فائیٹ تیاری لی کوئٹہ نیوز فلائی ویٹ چیمپئن باکسر محمد وسیم ٹھویں فائیٹ کھیلنے تیاری کرلی محمد وسیم پاناما باکسر کارل",
    "ایشوریا رائے مشکل میں نسل پرستی الزام لگ گیا ممبئی سابق ملکہ حسن ایشوریا رائے پر نسل پرستی الزام لگ گیا کمزور لاغر غلام بچے کی تصویر کے اشتہاری مہم نظ",
    "فلم کہانی گیس اسٹیشن کام کرنے والی لڑکی گرد گھومتی ہالی وڈ جون گیس اسٹیشن کام کرنے والی لڑکی جنونی قاتل بچنا چاہ لیکن اٹھتا قدم اسے موت جانب لے جاتا د",
    "سام سنگ طاقتور ٹیبلیٹس متعارف سام سنگ اس سال موبائل ورلڈ کانگریس اپنی گلیکسی ایس سیریز پیش نہیں کیا تاہم اس کمپنی اپنے طاقتور ٹیبلیٹس متعارف کرا ہے گل",
    "تاجروں رکنی وفد اس ماہ پاکستان ائے اسلام اباد پاکستان سفیر خصوصی جاوید ملک قیادت تاجروں دس رکنی بین الاقوامی وفد اس ماہ پاکستان جائے دبئی پاکستان سفیر",
    "ہالی ووڈ سنسنی خیز ڈرانی فلم بی فور ئی ویک کا ٹریلر جاری کیا ননھے بچے خواب سچ ہو کر کردیں والدین زندگی خراب اسی تجسس بنی ہے ہالی ووڈ سپر نیچرل ہارر فل",
    "امریکا چاند بیس بنائے گا واشنگٹن جنوری امریکا چاند بیس بنائے گا جبکہ ایک خلائی اسٹیشن چندا ماموں مدار میں گھومتا گذشتہ سال چاند انسان قدم نہیں رکھا ام",
    "رنبیر کپور مثالی شوہر قرار ممبئی ہندوستانی خواتین اکثریت اداکار رنبیر کپور مثالی شوہر قرار ہے ہندوستانی میڈیا ایک حالیہ سروے ہندوستانی نوجوان لڑکیوں خ",
    "بولڈ سینز کرنا نہ کرنا میری ذاتی مرضی پاکستانی اداکار فواد خان کہنا بولڈ سین کرنا نہ کرنا ان ذاتی مرضی ہے انڈیا مصالحہ میگزین انٹرویو فواد خان کہنا ان",
    "ائندہ بجٹ معیشت بہتری اقدامات کیے جائیں اسد عمر اسلام اباد مئی سابق وزیر خزانہ اسد عمر کہتے ملکی معیشت حالات ٹھیک نہیں بہتری مشکل فیصلے کرنا پڑیں گے ا",
    "نوکیا فولڈ ایبل فون بنانے فیصلہ نوکیا جانب ایک بار صارفین سرپرائز دیئے جانے امکان اس جانب غیرمعمولی اسمارٹ فون جلد سامنے ہے فن لینڈ تعلق رکھنے اس کمپن",
    "اے سی کے ساتھ جانے والی غلطیاں موسم گرما ائیر کنڈیشنر استعمال بڑھ جاتا ماہ بجلی بل ملتا پارہ چڑھنے لگتا ہے مگر اس وجہ اے سی استعمال نہیں بلکہ اپنی غلط",
    "فلم فیئر گلیمر ایوارڈز رنگا رنگ تقریب بولی وڈ ایک رنگا رنگ شام فلم فیئر گلیمر ایوارڈ ستاروں شرکت کر کے اسے یادگار بنا دیا اس دوران ریکھا سری دیوی دپیک",
    "خیبرپختونخواہ بجٹ پیش کردیا گیا پشاور اکتوبر خیبرپختونخواہ اٹھ ماہ بجٹ پیش کردیا گیا بجٹ اجلاس اپوزیشن واک اٹ کردیا پیپلز پارٹی اراکین بازوں سیاہ پٹیا",
    "ایران پر بین الاقوامی پابندیوں کا خاتمہ مشرق وسطی میں حصص کی منڈیوں میں کریش تہران ویب ڈیسک ایران بین الاقوامی پابندیوں خاتمے بعد مشرق وسطی حصص تمام م",
    "اسپیس ایکس کیپسول واپسی امریکا انسانی خلائی پروازوں کی راہ ہموار واشنگٹن اسپیس ایکس کریو ڈریگن کیپسول امریکی خلائی ادارے نیشنل اسٹرونوٹ اینڈ اسپیس ایڈ",
    "گردشی قرضے دوبارہ ارب روپے سے تجاوز کر گئے وزارت پانی بجلی پارلیمنٹ پبلک اکانٹس کمیٹی بتایا موجودہ حکومت ارب روپے زیر گردش قرضے سرکولر ڈیٹ کلئیر کیے گ",
    "یوایس اوپن امریکا کی سرینا ولیمز اور جاپان کی اوساکا فائنل میں پہنچ گئیں نیو یارک نیوز یوایس اوپن خواتین مقابلوں فائنلسٹ فیصلہ ہو گیا امریکا سرینا ولی",
    "محمد عامر میں سر اینڈی رابرٹس کی جھلک نظر آتی ہے کوچ سمرسیٹ کانٹی لندن ویب ڈیسک انگلش کاونٹی سمر سیٹ کوچ میتھیو مینرڈ پاکستانی باولرز گن گانے لگے کہتے",
    "محمد عامر پر جرمانہ عائد فیصل آباد فاسٹ بالر محمد عامر ڈومیسٹک کرکٹ میچ دوران ڈسپلن ورزی پر جرمانہ عائد گیا ہے قائد اعظم ٹرافی فرسٹ کلاس کرکٹ غاز سے ق",
    "کلبھوشن یادیو کا فیصلہ ارمی چیف میرٹ پر کریں گے ڈی جی ائی ایس پی آر راولپنڈی نیوز بھارتی جاسوس حوالے ترجمان پاک فوج میجر جنرل اصف غفور کہا کہ کلبھوشن ",
    "رواں برس تنقید اور تنازعات کی زد میں رہنے والے پاکستانی اسٹارز شوبز انڈسٹری وابستہ شخصیات خبروں زینت بنی رہتی لیکن ضروری نہیں خبریں مثبت ہوں تنازعات ت",
    "اداکارہ منال خان کے یورپ میں سیر سپاٹے پاکستان ڈرامہ انڈسٹری ابھرتی اداکارہ منال خان ان دنوں چھٹیاں منانے یورپ میں سیر ہیں اداکارہ منال خان اپنی چھٹیا",
    "بھارت کو شکست پاکستان نے ایشین ٹیم اسنوکر چیمپیئن شپ جیت لی پاکستان ایشین ٹیم اسنوکر چیمپیئن شپ فائنل میں بھارت کو شکست دے چیمپیئن بنے کا اعزاز حاصل ل",
    "سام سنگ کے نئے فلیگ شپ فون کا وہ فیچر جو ئی فون میں نہیں سام سنگ رواں سال دوسرا فلیگ شپ فون گلیکسی نوٹ اگلے ماہ متعارف کرایا جا رہا مگر جنوبی کورین کم",
    "پنجاب بجٹ اعداد شمار ملازمین کی تنخواہ میں اضافے کا امکان لاہور پنجاب بجٹ اعداد شمار حتمی شکل دیدی گئی سرکاری ملازمین تنخواہوں میں اضافے کا امکان مزدو",
    "پشاور جدید ٹیکنالوجی سے لیس کرائم سین پروٹیکشن یونٹ قائم پشاور جنوری پولیس پہلی مرتبہ اغوا قتل دیگر سنگین جرائم جائے وقوعہ پہنچنے کیلئے جدید ٹیکنالوجی",
    "سام سنگ کا پہلا فولڈ ایبل فون ایک ہفتے کی دوری پر سام سنگ اپنے فولڈ ایبل فون کا حقیقی ڈیزائن کی پہلی جھلک اگلے ہفتے سامنے لا رہی ہے جنوبی کورین کمپنی ",
    "اب فیس بک کے ساتھ جادو کرے گی فیس بک جلد جادو کر سکیں گی کمال رٹی فیشل ٹیکنالوجی اے ئی مبنی نظام کے ذریعے ممکن ہوگا جی ہاں فیس بک محققین ایک نئے اے ئی",
    "کوئٹہ کے مقامی کھلاڑی لاہور پہنچ گئے غیرملکی کرکٹرز کل پہنچیں گے لاہور پاکستان سپر لیگ پلے میچز کھیلنے کوئٹہ گلیڈی ایٹرز مقامی کھلاڑی کراچی لاہور پہنچ",
    "لندن ٹینس لیجنڈ بورس بیکر دیوالیہ ہو گئے لندن ویب ڈیسک ٹینس لیجنڈ بورس بیکر دیوالیہ ہو گئے بورس بیکر کے ذمہ عرصہ دراز سے بینک قرض واجب الادا تھا تفصیل",
    "شارجہ ٹیسٹ پاکستان اور انگلش ٹیمیں فٹنس مسائل سے دوچار شارجہ پاکستان انگلینڈ در میان اتوار شروع ہونے والے تیسرے اخری ٹیسٹ میچ قبل دونوں ٹیموں کے کھلاڑ",
    "سہیل تنویر ووسٹرشائر میں شامل دبئی سہیل تنویر اپنے ساتھی کھلاڑیوں سے پریکٹس کرتے ہوئے اے ایف پی کراچی انگلش کانٹی ووسٹر شائر پاکستان اسپنر سعید اجمل ک",
    "تھری ڈی پرنٹر نے دیو قامت کشتی چھاپ دی واشنگٹن اکتوبر گزشتہ ہفتے یونیورسٹی اف مین امریکا ایڈوانسڈ اسٹرکچرز اینڈ کمپوزٹس سینٹر میں نصب دنیا کے بڑے تھری",
    "میک بک پرو میں اچانک آگ بھڑک اٹھی نئی دہلی بھارتی شہری کا میک بک میں اچانک آگ بھڑک اٹھی جس کی ویڈیو اس نے ٹوئٹر پر شیئر کی صارف اپنے میک بک پرو پر روز",
    "سماجی تحفظ کے لیے بجٹ دگنا کرنے کا فیصلہ وفاقی حکومت احساس پروگرام کے تحت سماجی تحفظ کا مختص بجٹ دگنا کرنے فیصلہ کرلیا اسلام آباد پریس کانفرنس میں احس",
    "فیس بک ملازمین کا امریکی صدر کی پوسٹس پر کارروائی کا مطالبہ فیس بک ملازمین نے چیف ایگزیکٹو مارک زکربرگ کی جانب امریکی صدر ڈونلڈ ٹرمپ کی پوسٹس پر سخت ا",
    "پی سی بی کرکٹ کمیٹی کے چیئرمین اقبال قاسم کا استعفی منظور پاکستان کرکٹ بورڈ پی سی بی کرکٹ کمیٹی کا سربراہ اقبال قاسم کا استعفی منظور کر کے کہا ان کا م",
    "کترینا کیف نے سلمان خان کو پھر ٹھکرا دیا کترینا کیف کے سلمان خان کے پرستار خوش ہوگئے دونوں ایک فلم میں کام کرنے جا رہے تھے لیکن کیٹ بے بی نے مصروفیت ک",
    "سلمان کی فلمیں ہٹ پر ہٹ رنبیر کپور مسلسل ناکام ممبئی سلمان خان ایک بعد ایک سپر ہٹ فلمیں دے رہے لیکن دوسری طرف ان کے سابق گرل فرینڈ کے موجودہ فرینڈ رنب",
    "ای سی سی نے احساس پروگرام کے تحت ارب روپے جاری کرنے کی منظوری دے دی اقتصادی رابطہ کمیٹی ای سی سی نے احساس پروگرام کے تحت ارب روپے جاری کرنے کی منظوری ",
    "ٹی سی پی نے ہزار ٹن سستی چینی یوٹیلٹی اسٹورز کو فراہم کردی اسلام آباد ماہ رمضان میں فراہمی بہتر بنانے کیلئے ٹی سی پی نے ہزار ٹن سستی چینی یوٹیلٹی اسٹو",
    "پاکستان جنوبی افریقہ کے سامنے بے بس اتوار بلوم فونٹین میں کھیلے گئے ایک روزہ میچ میں جنوبی افریقہ نے پاکستان کو ایک سو پچیس رنز سے شکست دے کر سیریز می",
    "مہنگائی کے باعث فروٹ چاٹ اور پکوڑے جیب پر بھاری پڑ گئے کراچی اس بار ماہ رمضان میں فروٹ چاٹ پکوڑے جیب پر کافی بھاری پڑے سرکاری اعداد شمار میں کیلے چنے ",
    "ویرات کوہلی اور انوشکا شرما کی منگنی کی اطلاعات بھارتی ٹیسٹ کرکٹ ٹیم کے کپتان ویرات کوہلی اور بالی ووڈ اداکارہ انوشکا شرما کی منگنی کی اطلاعات کے بعد ",
    "معروف گلوکار عالمگیر نے موت کی افواہیں مسترد کردیں ماضی کے معروف گلوکار عالمگیر نے سوشل میڈیا پر وفات کی افواہوں کو مسترد کرتے ہوئے مداحوں کو زور سے ک",
    "جاپان میں مزدوروں کی کمی دور کرنے کیلیے مستری روبوٹ تیار ٹوکیو اکتوبر جاپان میں شرح پیدائش کم اور مزدوروں کی شدید قلت دیکھتے ہوئے ایک مزدور روبوٹ تیار",
    "سوائن فلو کے دو مشتبہ مریضوں کی ہلاکت فائل تصویر فلو کے خلاف ایک ویکسین تیار جا رہی ہے فائل راولپنڈی منگل رات ہولی فیملی ہسپتال میں انتقال کرنے والے د",
    "برائن ویٹوری ون ڈے میں دنیا کے چوتھے مہنگے ترین بالر نیپیئر زمبابوے کرکٹ ٹیم کے میڈیم فاسٹ بالر برائن ویٹوری ون ڈے کرکٹ میں دنیا کے چوتھے مہنگے ترین ب",
    "کراچی سے خیبر تک پیٹرول کا مصنوعی بحران جاری کراچی جون کراچی سے خیبر تک جاری مصنوعی بحران کا دسویں روز داخل ہوگیا تفصیلات میں کراچی لاہور پشاور سمیت م",
    "پاکستان نے کینیڈا کو اذلان شاہ ہاکی ٹورنامنٹ میں ہرا کر فاتحانہ غاز کر دیا کوالالمپور ویب ڈیسک پچیسویں اذلان شاہ ہاکی ٹورنامنٹ میں پاکستان نے فاتحانہ ",
    "بنگلہ دیش کو سری لنکا کے خلاف جیت کیلئے رنز مزید درکار بنگلہ دیش سری لنکا سیریز کے ٹیسٹ کے چوتھے روز رنز کا ہدف تعاقب میں نقصان رنز بنا کر اسے فتح کیل",
    "کورونا وائرس سے یورپ میں شٹ ڈان پاکستان کی ٹیکسٹائل برامدات متاثر کراچی یورپی خریداروں نے پاکستانی ٹیکسٹائل برامدکنندگان سے رابطہ کر کے ان سے درخواست ",
    "گلوکار علی ظفر کے ہے سال ہوگئے تھوڑے عرصے میں شہرت کی بلندیوں کو چھو جانے والے گلوکار موسیقار نغمہ نگار اداکار ماڈل پینٹر علی ظفر آج ویں سالگرہ مناہے ",
    "دپیکا پڈوکون سلمان خان کے ساتھ فلم میں کام کرنے کی خواہشمند ممبئی نے جی ٹی بالی ووڈ صف اول کی اداکاراں میں شامل دپیکا پڈکون نے سلمان خان کے ساتھ فلم م",
    "پاکستان اور انگلینڈ ٹیسٹ سیریز دوسرا ٹیسٹ کل سے دبئی میں شروع ہو رہا ہے پاکستان اور انگلینڈ کی کرکٹ ٹیمیں دوسرے پانچ روزہ معرکے کیلئے تیار ہیں دوسرا ٹ",
    "شاہ رخ خان پھر بنیں گے کترینہ کے ہیرو کنگ خان اور کترینہ کیف کو پھر سے جوڑی میں بنائیں گے فلمساز ادتیہ چوپڑا نے رومینٹک فلم کے لیے رابطے تیز کردیئے نج",
    "نیپرا نے فیول ایڈجسٹمنٹ کی مد میں فی یونٹ روپے اضافہ کردیا اسلام آباد بجلی کے نہ کے پاورپلانٹس میں فیول ملے نہ ملے عوام کو فیول ایڈجسٹمنٹ دینا پڑے جی ",
    "انٹرنیٹ ایکسپلورر کا دور ختم ہونے کے قریب مائیکروسافٹ نے اپنے انٹرنیٹ برازر انٹرنیٹ ایکسپلورر کے پرانے ورژنز کا سپورٹ فراہم کرنا بند کردیا ہے مائیکروس",
    "موبائل کو کمپیوٹر پر ترجیح فائل فوٹو واشنگٹن دنیا بھر کے لوگ موبائل فون کے ذریعے انٹرنیٹ رسائی حاصل کرتے ہیں غیرملکی خبر رساں ایجنسی کی رپورٹ میں موبا",
    "پلوامہ حملے سے متعلق بیان سدھو کو کپل شرما شو سے ہاتھ دھونے پڑے نئی دہلی بھارتی ریاست پنجاب کے وزیر سابق کرکٹر نوجوت سنگھ سدھو کے پلوامہ حملے سے متعلق",
    "واٹس ایپ پر غیرمتعلقہ پیغامات روکنے کیلئے نئے فیچر کی زمائش سوشل میڈیا کا مشہور ایپ واٹس ایپ جعلی افواہوں اور غیر متعلقہ اسپام پیغامات روکنے کا نیا فی",
    "ایڈونچر سے بھرپور فلم اسیسنز کریڈ کا ٹریلر جاری معروف وڈیو گیم مبنی ایکشن ایڈونچر سے بھرپور ہالی وڈ فینٹسی فلم اسیسنز کریڈ کا ٹریلر جاری گیا فلم یکم د",
    "پی ایس ایل میں آج لاہور قلندرز اور پشاور زلمی ٹکرائیں گے پاکستان سپر لیگ میں آج لاہور قلندرز اور پشاور زلمی کے درمیان مقابلہ ہے دونوں ٹیموں کو ایک میچ",
    "جنوبی افریقہ کا تجارتی وفد آل اپریل کو پاکستان پہنچے گا دورے میں کئی ملین ڈالر کی مالیت کے تجارتی کاروباری معاہدے متوقع رفیق میمن فوٹو فائل کراچی جنوب",
    "نیوزی لینڈ میں کیرئیر کا اخری میچ یادگار بنانا چاہتا ہوں افریدی ویلنگٹن پاکستان کی ٹی ٹیم کے کپتان ال رائونڈر شاہد افریدی نے کہا نیوزی لینڈ میں اپنے ک",
    "ایشوریہ رائے بچن کی انتالیسویں سالگرہ منا رہی ہیں ممبئی بھارتی اداکارہ ایشوریہ رائے بچن کی انتالیسویں سالگرہ منا رہی ہیں بالی ووڈ میں کئی کامیاب فلمیں",
    "ایپل منفرد ائی فون بنانے کا خواہشمند پڑھیں ایپل ایک منفرد لیپ ٹاپ بنانے کا خواہشمند مزید پڑھیں ایپل کا بہترین ئی فون ایکس معروف امریکی ٹیکنالوجی کمپنی",
    "ڈیجیٹل گولڈ کی قیمت میں ریکارڈ اضافہ عرصہ سے سونے کی قیمت میں کمی اور اضافہ اہم خبر ہوتی ہے مگر لگتا ہے یہ اعزاز ڈیجیٹل گولڈ یعنی بٹ کوائن کو چلا گیا ",
    "سیف علی خان سوشل میڈیا پر موجود نہیں بولی وڈ کے اداکار اپنے مداحوں سے رابطے میں رہنے کیلئے سوشل میڈیا پر کافی سرگرم رہتے ہیں تاہم انٹرٹینمنٹ انڈسٹری م",
    "گلیکسی نوٹ کو بھلا دینے والا منفرد اسمارٹ فون ویوو کمپنی جس کا اپنا کانسیپٹ فون نیکس متاثر کن اسکرین ٹو باڈی ریشو والا ہے اس سیریز کا تیسرا فون جلد مت",
    "فرنچ اوپن میں جووکووچ اور سمانتھا اسٹوزر دوسرے رانڈ میں پیرس میں ٹاپ سیڈ نوواک جووکووچ اور سابق فائنلسٹ سمانتھا اسٹوزر فرنچ اوپن ٹینس ٹورنامنٹ میں دوس",
    "حکومت نے سال میں بروقت ضروری اور تعمیراتی صنعت کو کھولا حماد اظہر اسلام آباد وفاقی وزیر برائے صنعت اور پیداوار حماد اظہر نے کہا پی ٹی ئی حکومت کی دو س",
    "انسٹاگرام کی فیڈ سروس متاثر صارفین کی شکایت پر کمپنی کی وضاحت موبائل فون میں انسٹاگرام ایپ کا ہوم فیڈ اسکرول کرنے میں مشکلات کا سامنا اس کا ہرگز مطلب ",
    "فرانس میں ویں انٹرسلٹیک فیسٹیول کا اغاز فرانس نیوز فرانس میں ویں انٹرسلٹیک فیسٹیول کا اغاز ہوگیا جس کی افتتاحی تقریب میں ستر ہزار افراد کے بھرپور شرکت",
    "ملک بھر میں سونے کی قیمت میں کمی ملک بھر میں سونے کی قیمت میں دو سو روپے فی تولہ کمی ہوگئی جس کے بعد فی تولہ سونے کی قیمت تریپن ہزار ایک سو روپے ہوگئی",
    "اسکر کی تقریب کے دوران بارش کا پانی چھتریوں سے بہہ نکالا ہالی ووڈ کیلیفورنیا اسکر کی تقریب کے دوران بارش کا پانی چھتریوں سے بہہ نکالا ریڈ کارپٹ محفوظ ",
    "چین نے سیٹلائٹ مدار میں روانہ کردیا زمین کا سروے کرے گا بیجنگ جون چین نے سیٹلائٹ کو مدار میں روانہ کردیا جو سیارجہ زمین کا سروے کرے گا بلیٹ روڈ اور دی",
    "ورلڈ ونڈ انرجی کانفرنس اور نمائش آج سے کراچی میں شروع ہوگی کراچی نومبر ورلڈ ونڈ انرجی کانفرنس اور نمائش کراچی میں آج شروع ہوگی کانفرنس میں دنیا بھر سے",
    "واٹس ایپ ایپلیکشن پرانے اپریٹنگ سسٹمز پر بند ویب ڈیسک فروری میں حاضر پیغام رسانی کا موثر اور اہم ذریعہ واٹس ایپ کی سروس لاکھوں پرانے فونز پر بند جا رہ",
    "پی ایس ایل فائنل کے یادگار لمحات پاکستان سپر لیگ فائنل میں کوئٹہ گلیڈی ایٹرز اور پشاور زلمی کے درمیان لاہور میں کھیلا گیا جہاں کامیابی پشاور زلمی کے ن",
    "حفیظ شیخ اور رزاق داد سے اختلافات چیئرمین سرمایہ کاری بورڈ زبیر گیلانی مستعفی چیئرمین سرمایہ کاری انویسمنٹ بورڈ زبیر گیلانی نے اپنے عہدے سے استعفی دے "
]
    # Execute all queries
    for idx, query in enumerate(test_queries, 1):
        print("\n" + "="*80)
        print(f"PROCESSING QUERY {idx} OF {len(test_queries)}")
        print("="*80)

        # Get recommendations
        results = recommender.get_recommendations(
            query=query,
            n_results=15  # 15 recommendations per query
        )

        # Display recommendations
        recommender.display_recommendations(results, query, query_number=idx)

    # Save the final document
    recommender.save_document()

    print("\n" + "="*80)
    print("✓ RECOMMENDATION SYSTEM COMPLETED!")
    print(f"✓ PCA embeddings: 128 dimensions")
    print(f"✓ Pooling method: Mean Pooling")
    print(f"✓ Content source: ChromaDB Metadata")
    print(f"✓ All {len(test_queries)} queries processed")
    print(f"✓ Report saved to: {recommender.output_doc_path}")
    print("="*80)

URDU NEWS RECOMMENDATION SYSTEM
128D PCA Embeddings with Mean Pooling
Content Retrieved from ChromaDB Metadata

INITIALIZING RECOMMENDATION SYSTEM

Using device: cuda
Loading model: urduhack/roberta-urdu-small
Loading PCA model from: chroma_db_collections/chroma_db_pca_128D/pca_model.pkl
✓ PCA model loaded (128 dimensions)
Connecting to ChromaDB at: chroma_db_collections/chroma_db_pca_128D
✓ Connected to collection: urdu_news_pca_128D
✓ Total articles in database: 111853


SYSTEM STATISTICS
Model: urduhack/roberta-urdu-small
Device: cuda
Pooling Method: MEAN POOLING
Embedding Dimension: 128 (PCA Reduced)
PCA Explained Variance: 96.66%
Total Articles: 111853
Collection Name: urdu_news_pca_128D
Content Source: ChromaDB Metadata


PROCESSING QUERY 1 OF 100

GENERATING RECOMMENDATIONS (128D PCA)
Query: ریاضی اس سوال جواب دے پہلی نظر ریاضی سان سوال لگتا مگر اس انٹرنیٹ متعدد افراد ذہنوں پریشان کرکے رکھ ...
Number of results: 15

→ Generating query embedding (128D PCA with Mean Pooling)...
  